# Activities, peaks, and exposure statistics

This example shows how to:

- Load a 1D time series (CPC total number concentration)
- Inspect main and extra data
- Define simple time segments (activities)
- Detect peaks using `peak_finder`
- Plot total concentration with activity shading
- Summarize activities and compute exposure metrics with `summarize_exposure`

In [ ]:
import matplotlib.pyplot as plt
import aerosoltools as at

## Load CPC sample data

Here we load a sample CPC file from the test data folder.
Adjust the path if you keep example data elsewhere.

In [ ]:
filename = "../../tests/data/Sample_CPC_AIM.txt"
cpc = at.load_cpc_file(filename,extra_data=True)

## Inspect metadata and extra data

The loader returns an `Aerosol1D` instance with:

- `.data` – main time series + activity masks
- `.extra_data` – optional additional channels (environmental, meta, etc.)
- `.metadata` – instrument, unit, etc.

In [ ]:
print("Metadata:\n", cpc.metadata)
print("\nMain columns:", list(cpc.data.columns))
print("Extra-data columns:", list(cpc.extra_data.columns))

You can access data in the main DataFrame via the `.data` property of the defined cpc variable:

In [ ]:
cpc.data


Similarly, you can access any additional data columns extracted from the raw data via the `.extra_data` property:

In [ ]:

cpc.extra_data

## Plot total concentration

Use `plot_total_conc()` to quickly visualize the time series.

In [ ]:
cpc.plot_total_conc()

## Detect peaks with `peak_finder`

`peak_finder` flags time steps where the signal exceeds a rolling baseline
by more than `ratio * rolling_std`. The resulting mask is stored as the
`"Peak"` activity.

In [ ]:
cpc.peak_finder(window=15, ratio=2.5, method="median")
cpc.activities

We can also mark specific activities based on timestamps:

In [ ]:
activity_periods= {
    "initial_phase": [
        ("2023-08-14 11:14:00", "2023-08-14 11:17:00"),
        ("2023-08-14 11:17:30", "2023-08-14 11:18:00")],
    "second_phase": [
        ("2023-08-14 11:19:00", "2023-08-14 11:20:00")]
}
cpc.mark_activities(activity_periods)

## Plot with activity shading

We can overlay activities (including the automatically created `"Peak"` mask)
on top of the total concentration time series.

In [ ]:
cpc.plot_total_conc(mark_activities=True)

## Summarize activities

`summarize_activities()` reports basic descriptive statistics per activity,
including duration, mean, median, and number of samples.

In [ ]:
cpc.summarize_activities()

## Exposure summary for PNC

Finally, we use `summarize_exposure` on the `Aerosol1D` object to compute
exposure metrics for total number concentration (PNC):

- duration, mean, percentiles
- time above a long-term limit
- short-term (window-based) exceedances
- an 8-hour TWA (by default)

In [ ]:
exp = cpc.summarize_exposure(
    metric="PNC",           # use total_concentration
    activities=["All data"],  # summarize whole record
    background=None,        # assume zero background outside the record
    exposure_hours=8.0,     # Set how long a worker was exposed to the measured concentration
    short_limit=1.0,        # Short term exposure limit for the given metric (if available)
    long_limit=1.0,         # long term or 8hr exposure limit for the given metric
    short_window="15min",   # Length of short term exposures to identify
)
exp